# LF7 — Đánh nhãn độ hữu dụng bằng vision model (Qwen3-VL)

Mỗi ảnh được model thị giác chấm `suitable` + `confidence` cho từng tác vụ, quy thành phiếu
`1` (hữu dụng) / `0` (không) / `abstain` theo ngưỡng τ.

## Cài đặt

In [ ]:
# Chỉ nâng transformers khi quá cũ, dùng --no-deps để KHÔNG đụng torch/pillow của Kaggle.
import torch
import transformers
from packaging.version import parse as V

assert torch.cuda.is_available(), 'CUDA=False — torch-CUDA đã bị thay. Factory reset session rồi chạy lại.'

if V(transformers.__version__) < V('4.57'):
    %pip install -q -U --no-deps transformers huggingface_hub
    print('Đã nâng transformers — Restart kernel rồi chạy lại từ đầu.')

## Cấu hình

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from PIL import Image, ImageOps

HF_MODEL = 'Qwen/Qwen3-VL-4B-Instruct'
KAGGLE_MODEL_DIR = '/kaggle/input/models/qwen-lm/qwen-3-vl/transformers/4b-instruct/1'
KAGGLE_DATASET = 'duongthimyphuong/coconut'
SEED = 42

MAX_LONG_EDGE = 768
MAX_NEW_TOKENS = 384
BATCH_SIZE = 8

TAU_HIGH = 0.70
TAU_LOW = 0.30

N_PER_SOURCE = 8

GATE_PROC_DIM = 512
GATE_MIN_DIM = 200
GATE_BLUR_MIN = 60.0
GATE_DARK_MAX = 0.60
GATE_BRIGHT_MAX = 0.45
GATE_MEAN_MIN = 25
GATE_MEAN_MAX = 235

TASKS = [
    '1_maturity_evaluation',
    '2_foliar_disease',
    '3_trunk_disease',
    '4_crown_disease',
    '5_petiole',
]
FIELD = {
    '1_maturity_evaluation': 'maturity_evaluation',
    '2_foliar_disease': 'foliar_disease',
    '3_trunk_disease': 'trunk_disease',
    '4_crown_disease': 'crown_disease',
    '5_petiole': 'petiole',
}
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if Path('/kaggle').exists():
    runner = 'Kaggle'
elif Path('/content').exists():
    runner = 'Google Colab'
else:
    runner = 'Local'

if runner == 'Kaggle':
    MODEL = KAGGLE_MODEL_DIR
else:
    MODEL = HF_MODEL

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_count = torch.cuda.device_count()
else:
    gpu_name = 'CPU'
    gpu_count = 0

print('Model        =', MODEL)
print('Runner       =', runner)
print('GPU          =', gpu_name, 'x', gpu_count)
print('transformers =', transformers.__version__)
print('torch        =', torch.__version__, '| CUDA', torch.cuda.is_available())

## Dữ liệu — đường dẫn dataset (local / Kaggle)

Hai hàm dùng chung cho mọi LF: `dataset_paths_local()` và `dataset_paths_kaggle()`.
Cùng gọi helper `_resolve_dataset()` (ghép sub-path + kiểm tra tồn tại). Khác nhau ở BASE và độ lồng:
local single-nest dưới `Dataset/`, Kaggle double-nest dưới `/kaggle/input/datasets/<owner>/coconut/`.

In [ ]:
CLASSES = [
    'Gray Leaf Spot',
    'Leaf Rot',
    'Stem Bleeding',
    'Bud Rot',
    'Bud Root Dropping',
]


def _resolve_dataset(base, disease_rel, veirf_rel):
    disease_dir = base / disease_rel
    veirf_dir = base / veirf_rel
    if not disease_dir.is_dir():
        raise SystemExit(f'Không thấy {disease_dir} — kiểm tra lại đường dẫn dataset.')
    if not veirf_dir.is_dir():
        raise SystemExit(f'Không thấy {veirf_dir} — kiểm tra lại đường dẫn dataset.')
    return disease_dir, veirf_dir, base


def dataset_paths_local():
    base = None
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / 'Dataset').is_dir():
            base = parent / 'Dataset'
            break
    if base is None:
        raise SystemExit('Không thấy thư mục Dataset/ ở local — chạy notebook trong repo chứa Dataset/.')
    disease_rel = Path('Coconut Tree Disease Dataset')
    veirf_rel = Path('coconut-veirf-v5')
    return _resolve_dataset(base, disease_rel, veirf_rel)


def dataset_paths_kaggle():
    base = Path('/kaggle/input/datasets') / KAGGLE_DATASET
    disease_rel = Path('Coconut Tree Disease Dataset') / 'Coconut Tree Disease Dataset'
    veirf_rel = Path('coconut-veirf-v5') / 'coconut-veirf-v5'
    return _resolve_dataset(base, disease_rel, veirf_rel)


if runner == 'Kaggle':
    DISEASE_DIR, VEIRF_DIR, DATA_ROOT = dataset_paths_kaggle()
else:
    DISEASE_DIR, VEIRF_DIR, DATA_ROOT = dataset_paths_local()

if runner == 'Kaggle':
    OUT_DIR = Path('/kaggle/working/labels')
else:
    OUT_DIR = DATA_ROOT.parent / 'labels'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / 'lf7_vision_pilot_qwen.csv'

print('DISEASE_DIR =', DISEASE_DIR)
print('VEIRF_DIR   =', VEIRF_DIR)
print('OUT_CSV     =', OUT_CSV)

## Cổng chất lượng (tất định, chạy trước model)

In [ ]:
_LAP = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32)


def lap_var(gray):
    padded = np.pad(gray, 1, mode='reflect')
    out = np.zeros_like(gray)
    for dy in range(3):
        for dx in range(3):
            out += _LAP[dy, dx] * padded[dy:dy + gray.shape[0], dx:dx + gray.shape[1]]
    return float(out.var())


def quality_gate(path):
    img = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    w, h = img.size

    if max(w, h) > GATE_PROC_DIM:
        scale = GATE_PROC_DIM / max(w, h)
    else:
        scale = 1.0

    if scale < 1.0:
        small = img.resize((max(1, int(w * scale)), max(1, int(h * scale))))
    else:
        small = img

    arr = np.asarray(small, dtype=np.float32)
    gray = arr @ np.array([0.299, 0.587, 0.114], dtype=np.float32)
    lap = lap_var(gray)
    mean = float(gray.mean())
    dark = float((gray < 15).mean())
    bright = float((gray > 245).mean())

    reasons = []
    if min(w, h) < GATE_MIN_DIM:
        reasons.append('lowres')
    if lap < GATE_BLUR_MIN:
        reasons.append('blur')
    if dark > GATE_DARK_MAX or mean < GATE_MEAN_MIN:
        reasons.append('underexposed')
    if bright > GATE_BRIGHT_MAX or mean > GATE_MEAN_MAX:
        reasons.append('overexposed')

    return {
        'quality_pass': not reasons,
        'gate_reason': '|'.join(reasons),
        'lap_var': round(lap, 1),
        'mean_bright': round(mean, 1),
        'min_dim': min(w, h),
    }

## Rubric + schema

In [ ]:
from pydantic import BaseModel

SYSTEM = '''Bạn là giám định viên ảnh nông nghiệp. Với ảnh cây/trái dừa (đã qua bộ lọc chất lượng cơ bản),
hãy đánh giá ĐỘC LẬP cho từng tác vụ: ảnh có cho thấy rõ đối tượng/bộ phận cần thiết ĐỦ ĐỂ ĐÁNH GIÁ tác vụ đó hay không.
NGUYÊN TẮC: suitable=true khi ảnh đủ rõ để KẾT LUẬN — dù kết luận là CÓ hay KHÔNG có vấn đề. KHÔNG đòi hỏi ảnh phải đang có triệu chứng bệnh: một bộ phận KHỎE MẠNH nhưng hiện rõ, đủ nét vẫn suitable=true. Chỉ đặt suitable=false khi đối tượng cần thiết không hiện diện, bị che khuất, hoặc quá kém để đánh giá.
- maturity_evaluation: thấy rõ >=1 trái dừa với kích thước/hình dạng/màu vỏ tái hiện trung thực, đủ để đánh giá độ chín (dry/green/tender).
- foliar_disease: thấy rõ phiến lá/tàu lá đủ nét để đánh giá tình trạng mặt lá (có hay không đốm/cháy/thối).
- trunk_disease: thấy rõ bề mặt thân/gốc đủ nét để đánh giá tình trạng thân (có hay không chảy nhựa/loét/đổi màu).
- crown_disease: thấy được cụm ngọn/đọt trung tâm của cây dừa — nơi các tàu lá non và lá đọt (spear) mọc ra — kể cả nhìn ngang/chếch từ mặt đất (KHÔNG cần từ trên xuống) — đủ rõ để nhận xét bề mặt vùng đó (có hay không thối nhũn, úng nâu ở gốc bẹ non, gãy gục đọt, đổi màu). suitable=true khi thấy cụm ngọn/đọt đủ nét; suitable=false khi không thấy cụm đó, bị che, hoặc quá mờ.
- petiole: thấy được tổng thể tán hoặc phần ngọn cây dừa đủ để nhận xét tư thế tàu lá — vươn thẳng/xòe đều (khỏe) hay rủ gập, khô nâu, xẹp thành "váy" quanh thân (suy tàn). KHÔNG cần cận cảnh cuống lá. suitable=true khi thấy đủ nhiều tàu lá/ngọn để phán tư thế; suitable=false khi chỉ thấy lá chét lẻ, hoặc quá xa/mờ/khuất.
Nếu đối tượng cần thiết không hiện diện hoặc bị che khuất/kém quá mức cho một tác vụ -> suitable=false.
confidence là số thực 0..1 = mức độ ảnh ĐỦ RÕ để đánh giá tác vụ đó (KHÔNG phải xác suất có bệnh).'''


class Task(BaseModel):
    suitable: bool
    confidence: float
    reason: str


class ImageLabel(BaseModel):
    maturity_evaluation: Task
    foliar_disease: Task
    trunk_disease: Task
    crown_disease: Task
    petiole: Task

## Lấy mẫu ảnh

In [ ]:
def list_images(folder):
    if not folder.is_dir():
        return []
    files = [p for p in folder.iterdir() if p.suffix.lower() in IMG_EXTS]
    return sorted(files)


def sample_even(items, n):
    if len(items) <= n:
        return items
    step = len(items) / n
    return [items[int(i * step)] for i in range(n)]


def collect_sources():
    sources = []
    for cls in CLASSES:
        images = list_images(DISEASE_DIR / cls)
        if images:
            sources.append((cls, images))

    veirf_images = []
    for split in ['train', 'valid', 'test']:
        veirf_images += list_images(VEIRF_DIR / split / 'images')
    if veirf_images:
        sources.append(('coconut-veirf-v5', sorted(veirf_images)))

    return sources


sources = collect_sources()

pilot = []
for source_folder, images in sources:
    for path in sample_even(images, N_PER_SOURCE):
        pilot.append((path, source_folder))

print('Số nguồn   =', len(sources))
print('Số ảnh mẫu =', len(pilot))
pd.Series([folder for _, folder in pilot]).value_counts()

## Nạp model (Qwen3-VL 4B, FP16)

In [ ]:
import os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from transformers import AutoModelForImageTextToText, AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL)
processor.tokenizer.padding_side = 'left'
model = AutoModelForImageTextToText.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
    device_map='cuda',
)
model.eval()

print('Đã nạp', MODEL)
for i in range(torch.cuda.device_count()):
    used_gb = torch.cuda.memory_allocated(i) / 1e9
    print(f'GPU{i}: {used_gb:.1f} GB')

## Hàm chấm nhãn (theo batch)

In [ ]:
import json

USER_TEXT = (
    'Đánh giá độc lập ảnh dừa này cho từng tác vụ. '
    'CHỈ trả về JSON đúng khoá: maturity_evaluation, foliar_disease, trunk_disease, '
    'crown_disease, petiole; mỗi khoá là object '
    '{"suitable": bool, "confidence": số 0..1, "reason": chuỗi ngắn}. '
    'Không thêm chữ nào ngoài JSON.'
)
RETRY_HINT = '\nLần trước JSON sai định dạng. Trả về DUY NHẤT một object JSON hợp lệ.'


def load_image(path):
    img = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    w, h = img.size
    if max(w, h) > MAX_LONG_EDGE:
        scale = MAX_LONG_EDGE / max(w, h)
        img = img.resize((int(w * scale), int(h * scale)))
    return img


def extract_json(text):
    text = text.strip()
    if text.startswith('```'):
        text = text.strip('`')
    start = text.find('{')
    end = text.rfind('}')
    return json.loads(text[start:end + 1])


def try_parse(raw):
    try:
        return ImageLabel(**extract_json(raw))
    except Exception:
        return None


@torch.inference_mode()
def generate_batch(imgs, hint):
    prompts = []
    for _ in imgs:
        messages = [
            {'role': 'system', 'content': SYSTEM},
            {'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': USER_TEXT + hint}]},
        ]
        prompts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    inputs = processor(text=prompts, images=imgs, padding=True, return_tensors='pt').to(model.device)
    generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    trimmed = generated[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(trimmed, skip_special_tokens=True)


def label_batch(imgs):
    raws = generate_batch(imgs, '')
    results = [try_parse(r) for r in raws]

    retry_index = [i for i, r in enumerate(results) if r is None]
    if retry_index:
        retry_raws = generate_batch([imgs[i] for i in retry_index], RETRY_HINT)
        for j, i in enumerate(retry_index):
            results[i] = try_parse(retry_raws[j])
    return results


def vote(confidence):
    if confidence >= TAU_HIGH:
        return 1
    if confidence <= TAU_LOW:
        return 0
    return -1

## Chạy và lưu (batch, checkpoint từng batch + resume)

In [ ]:
import time


def chunks(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]


def base_row(path, source_folder, gate):
    return {
        'image_id': path.stem,
        'source_folder': source_folder,
        'path': str(path.relative_to(DATA_ROOT)),
        'quality_pass': gate['quality_pass'],
        'gate_reason': gate['gate_reason'],
        'lap_var': gate['lap_var'],
        'mean_bright': gate['mean_bright'],
        'min_dim': gate['min_dim'],
    }


def make_row(path, source_folder, gate, label):
    row = base_row(path, source_folder, gate)
    if label is None:
        if gate['quality_pass']:
            reason = 'parse_error'
        else:
            reason = ''
        for task in TASKS:
            row[f'lf7_{task}'] = -1
            row[f'{task}_conf'] = np.nan
            row[f'{task}_reason'] = reason
        return row
    for task in TASKS:
        task_result = getattr(label, FIELD[task])
        row[f'lf7_{task}'] = vote(float(task_result.confidence))
        row[f'{task}_conf'] = round(float(task_result.confidence), 3)
        row[f'{task}_reason'] = task_result.reason
    return row


done_ids = set()
if OUT_CSV.exists():
    done_ids = set(pd.read_csv(OUT_CSV)['image_id'].astype(str))
    print('Đã có', len(done_ids), 'ảnh — bỏ qua, chạy tiếp.', flush=True)

pending = []
for path, source_folder in pilot:
    if path.stem not in done_ids:
        pending.append((path, source_folder))
print('Còn lại', len(pending), 'ảnh', flush=True)

header_written = OUT_CSV.exists()
processed = 0
start = time.time()
for chunk in chunks(pending, BATCH_SIZE):
    gates = [quality_gate(path) for path, _ in chunk]

    passed_imgs = []
    passed_pos = []
    for pos, ((path, folder), gate) in enumerate(zip(chunk, gates)):
        if gate['quality_pass']:
            passed_imgs.append(load_image(path))
            passed_pos.append(pos)

    labels_by_pos = {}
    if passed_imgs:
        labels = label_batch(passed_imgs)
        for k, pos in enumerate(passed_pos):
            labels_by_pos[pos] = labels[k]

    rows = []
    for pos, ((path, folder), gate) in enumerate(zip(chunk, gates)):
        label = labels_by_pos.get(pos)
        rows.append(make_row(path, folder, gate, label))

    pd.DataFrame(rows).to_csv(OUT_CSV, mode='a', header=not header_written, index=False)
    header_written = True
    processed += len(rows)
    print(f'{processed}/{len(pending)}  ({time.time() - start:.1f}s)', flush=True)

print('Xong. File:', OUT_CSV, flush=True)
final = pd.read_csv(OUT_CSV)
columns = ['image_id', 'source_folder', 'quality_pass'] + [f'lf7_{t}' for t in TASKS]
final[columns].head(10)